In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow  

df = pd.read_excel("Sample - Superstore 2019.xls")
print("shape:", df.shape)
df.head()


ImportError: `Import xlrd` failed. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install the xlrd package.

In [ ]:
df.tail()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
9989,9990,CA-2016-110422,2016-01-21,2016-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,33180.0,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.248,3,0.2,4.1028
9990,9991,CA-2019-121258,2019-02-26,2019-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627.0,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.960,2,0.0,15.6332
9991,9992,CA-2019-121258,2019-02-26,2019-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627.0,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.576,2,0.2,19.3932
9992,9993,CA-2019-121258,2019-02-26,2019-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627.0,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.600,4,0.0,13.3200
9993,9994,CA-2019-119914,2019-05-04,2019-05-09,Second Class,CC-12220,Chris Cortes,Consumer,United States,Westminster,...,92683.0,West,OFF-AP-10002684,Office Supplies,Appliances,"Acco 7-Outlet Masterpiece Power Center, Wihtou...",243.160,2,0.0,72.9480


In [ ]:
df_raw = df.copy()
df_raw.shape

(9994, 21)

In [ ]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          9994 non-null   int64         
 1   Order ID        9994 non-null   str           
 2   Order Date      9994 non-null   datetime64[us]
 3   Ship Date       9994 non-null   datetime64[us]
 4   Ship Mode       9994 non-null   str           
 5   Customer ID     9994 non-null   str           
 6   Customer Name   9994 non-null   str           
 7   Segment         9994 non-null   str           
 8   Country/Region  9994 non-null   str           
 9   City            9994 non-null   str           
 10  State           9994 non-null   str           
 11  Postal Code     9983 non-null   float64       
 12  Region          9994 non-null   str           
 13  Product ID      9994 non-null   str           
 14  Category        9994 non-null   str           
 15  Sub-Category   

In [ ]:
print(df_raw.isnull().sum())

Row ID             0
Order ID           0
Order Date         0
Ship Date          0
Ship Mode          0
Customer ID        0
Customer Name      0
Segment            0
Country/Region     0
City               0
State              0
Postal Code       11
Region             0
Product ID         0
Category           0
Sub-Category       0
Product Name       0
Sales              0
Quantity           0
Discount           0
Profit             0
dtype: int64


In [ ]:
print("Duplicates:", df_raw.duplicated().sum())

Duplicates: 0


In [ ]:
# Drop unnecessary columns
def drop_unnecessary_columns(data):
    columns_to_drop = ["Row ID"]
    data = data.drop(columns=columns_to_drop)
    return data
df_raw = drop_unnecessary_columns(df_raw)

In [ ]:
df_raw.shape

(9994, 20)

In [ ]:
# Clean and standardize text columns (strip extra spaces and apply title case)
def clean_text_columns(data):
    string_cols = data.select_dtypes("str").columns.tolist()
    for col in string_cols:
        data[col] = data[col].str.strip().str.title()
    return data
df_raw = clean_text_columns(df_raw)

In [ ]:
df_raw["Segment"].unique()

<ArrowStringArray>
['Consumer', 'Corporate', 'Home Office']
Length: 3, dtype: str

In [ ]:
# Optimize data types (convert to datetime and category to save memory)
def convrert_data_types(data):
    date_cols = ["Order Date", "Ship Date"]
    for col in date_cols:
        if col in data.columns:
            data[col] = pd.to_datetime(data[col])
    str_cols = data.select_dtypes("str").columns.tolist()
    for col in str_cols:
        if len(data[col].unique()) <= 50:
           data[col] = data[col].astype("category")

    return data
df_raw = convrert_data_types(df_raw)

In [ ]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order ID        9994 non-null   str           
 1   Order Date      9994 non-null   datetime64[us]
 2   Ship Date       9994 non-null   datetime64[us]
 3   Ship Mode       9994 non-null   category      
 4   Customer ID     9994 non-null   str           
 5   Customer Name   9994 non-null   str           
 6   Segment         9994 non-null   category      
 7   Country/Region  9994 non-null   category      
 8   City            9994 non-null   str           
 9   State           9994 non-null   category      
 10  Postal Code     9983 non-null   float64       
 11  Region          9994 non-null   category      
 12  Product ID      9994 non-null   str           
 13  Category        9994 non-null   category      
 14  Sub-Category    9994 non-null   category      
 15  Product Name   

In [ ]:

missing_postal = df_raw[df_raw["Postal Code"].isnull()][["Country/Region", "City", "State", "Region"]]


print(missing_postal)

     Country/Region        City    State Region
2234  United States  Burlington  Vermont   East
5274  United States  Burlington  Vermont   East
8798  United States  Burlington  Vermont   East
9146  United States  Burlington  Vermont   East
9147  United States  Burlington  Vermont   East
9148  United States  Burlington  Vermont   East
9386  United States  Burlington  Vermont   East
9387  United States  Burlington  Vermont   East
9388  United States  Burlington  Vermont   East
9389  United States  Burlington  Vermont   East
9741  United States  Burlington  Vermont   East


In [ ]:
def handle_missing_data(data):
    if "Postal Code" in data.columns:
        data["Postal Code"] = data["Postal Code"].astype("str")
        data["Postal Code"] = data["Postal Code"].fillna("05401")

    return data
df_raw = handle_missing_data(df_raw)

In [ ]:
print(df_raw.isnull().sum())

Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country/Region    0
City              0
State             0
Postal Code       0
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
Quantity          0
Discount          0
Profit            0
dtype: int64


In [ ]:
def remove_duplicates(data):
    data = data.drop_duplicates()
    return data 
df_raw = remove_duplicates(df_raw)

In [ ]:
df_raw[["Sales", "Quantity", "Discount", "Profit"]].describe()

,Sales,Quantity,Discount,Profit
count,9993.000000,9993.000000,9993.000000,9993.000000
mean,229.852846,3.789753,0.156188,28.660971
std,623.276074,2.225149,0.206457,234.271476
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.280000,2.000000,0.000000,1.731000
50%,54.480000,3.000000,0.200000,8.671000
75%,209.940000,5.000000,0.200000,29.364000
max,22638.480000,14.000000,0.800000,8399.976000


In [ ]:
df_raw.describe(include="str")

,Order ID,Customer ID,Customer Name,City,Postal Code,Product ID,Product Name
count,9993,9993,9993,9993,9993,9993,9993
unique,5009,793,793,531,631,1862,1850
top,Ca-2019-100111,Wb-21850,William Brown,New York City,10035.0,Off-Pa-10001970,Staple Envelope
freq,14,37,37,915,263,19,48


In [ ]:
# Function 6: Sanity Checks (Logical Validation)
def sanity_checks(data):
    # Ensure discounts and quantities are within logical boundaries
    assert data["Discount"].min() >= 0 and data["Discount"].max() <= 1, "Error: Discount out of range (0 to 1)"
    assert data["Quantity"].min() > 0, "Error: Quantity cannot be zero or negative"
    
    print("Sanity checks passed successfully! All values are within logical bounds.")
sanity_checks(df_raw)

Sanity checks passed successfully! All values are within logical bounds.


In [ ]:
df_raw.to_parquet("Sample - Superstore 2019_clean.parquet", engine="pyarrow", index=False)